# Step-1> Problem Statement: Predicition of Sales

Rossmann operates over 3,000 drug stores in 7 European countries. Currently, Rossmann store managers are tasked with predicting their daily sales for up to six weeks in advance. Store sales are influenced by many factors, including promotions, competition, school and state holidays, seasonality, and locality. With thousands of individual managers predicting sales based on their unique circumstances, the accuracy of results can be quite varied.

The first step in any machine learning problem is to read the given documentation, talk to various stakeholders and identify the following:

1. What is the business problem you're trying to solve using machine learning?
2. Why are we interested in solving this problem? What impact will it have on the business?
3. How is this problem solved currently, without any machine learning tools?
4. Who will use the results of this model, and how does it fit into other business processes?
5. How much historical data do we have, and how was it collected?
6. What features does the historical data contain? Does it contain the historical values for what we're trying to predict.
7. What are some known issues with the data (data entry errors, missing data, differences in units etc.)
8. Can we look at some sample rows from the dataset? How representative are they of the entire dataset.
9. Where is the data stored and how will you get access to it?
10. ...

Gather as much information about the problem as possible, so that you're clear understanding of the objective and feasibility of the project.

In [ ]:
#%pip install opendatasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import opendatasets


##changing the scintific output of the pandas:
#     1)  pd.set_option('display.float_format', '{:.0f}'.format) #Disable scientific notation globally
#     2)  pd.set_option('display.float_format', '{:.2f}'.format) #Keep decimals but remove scientific notation
#     3) merged_df.describe().map(lambda x: f"{x:,.0f}")  #Only for spedific output (temporary)

#pd.set_option('display.float_format', '{:3f}'.format) not good

imporitng the data from kaggle

API KEY: KGAT_06e52b30bb2e2f9011f8bf473efe4f1b

HOW TO IMPORT:

export KAGGLE_API_TOKEN=KGAT_ef8feb3f15f129aaad3eb1b1a174c2f8

In [ ]:
#opendatasets.download('https://www.kaggle.com/c/rossmann-store-sales')
#opendatasets.download_kaggle_dataset('https://www.kaggle.com/c/rossmann-store-sales', data_dir=os.curdir)

# Step-2> is to import and understand the data

In [ ]:
ross_df = pd.read_csv(r'D:\Codes\Artificial_Intelligence\Machine_Learning\Machine-Learning-With-Scikit-Learn\Learning\datasets\rossmann\train.csv').copy()

In [ ]:
ross_df

In [ ]:
store_df = pd.read_csv(r'D:\Codes\Artificial_Intelligence\Machine_Learning\Machine-Learning-With-Scikit-Learn\Learning\datasets\rossmann\store.csv').copy()

In [ ]:
store_df.info()

In [ ]:
store_df.head(20)

In [ ]:
merged_df = ross_df.merge(store_df, how='left', on='Store')
merged_df


In [ ]:
merged_df.shape

In [ ]:
test_df = pd.read_csv(r'D:\Codes\Artificial_Intelligence\Machine_Learning\Machine-Learning-With-Scikit-Learn\Learning\datasets\rossmann\test.csv').copy()

In [ ]:
test_df

In [ ]:
merged_test_df = test_df.merge(store_df, how='left', on='Store')
merged_test_df

# Step-3> Cleaning Data

The first step is to check the column data types and identify if there are any null values

In [ ]:
merged_df.info()

In [ ]:
merged_df.isnull().sum()

In [ ]:
merged_df.describe().map(lambda x: f'{x:}')

In [ ]:
#check for duplicate rows also
merged_df.duplicated().sum()

In [ ]:
#parsing date
merged_df['Date'] = pd.to_datetime(merged_df.Date)
merged_test_df['Date'] = pd.to_datetime(merged_test_df.Date)

In [ ]:
merged_df.Date.min(), merged_df.Date.max()

In [ ]:
merged_test_df.Date.min(), merged_test_df.Date.max()


# Step-4> Exploratory Data Analysis and Vizualization

**Objectives of exploratory data analysis:**

    1) Study the distribution of individual columns(uniform, normal, exponential)

    2) Detect anomalies or errors in the data(e.g. missing/incorrect values)

    3) Study the relationship of target column with other columns(linear, non-linear etc.)

    4) Gather insights about the problem and the dataset

    5) Come up wit ideas for preprocessing and feature engineering


Study the distribution of the target 'Sales' column

In [ ]:
sns.histplot(data=merged_df, x='Sales')

In [ ]:
merged_df['Open'].value_counts()

To make our modeling simple, we are simply excluding the dates when the store was closed (we can handle it as a special case while making predictions)

In [ ]:
merged_df = merged_df[merged_df.Open==1].copy()

In [ ]:
sns.histplot(data=merged_df, x='Sales')

In [ ]:
temp_df = merged_df.sample(40000)
sns.scatterplot(data=temp_df, x='Sales', y='Customers', hue=temp_df.Date.dt.year, alpha=0.7)
plt.title('Sales Vs Customers')

In [ ]:
px.scatter(temp_df, x='Sales', y='Customers' ,color=temp_df.Date.dt.year, opacity=0.6)

In [ ]:
merged_df.columns

In [ ]:
px.scatter(merged_df.sample(40000), x='Store', y='Sales', color = merged_df.sample(40000).Date.dt.year, opacity=0.4)

In [ ]:
sns.barplot(data=temp_df, x='DayOfWeek', y='Sales')

In [ ]:
sns.barplot(data=temp_df, x='Promo', y='Sales')

In [ ]:
temp_df['Sales']

In [ ]:
merged_df.select_dtypes(include='number').corr()['Sales'].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=[15, 6])
sns.barplot(data=temp_df, x='Promo2SinceWeek', y='Sales')

# Step-5> Feature Engineering

Feature engineering is the process of creating new features (columns) by **transforming/combining** existing features or by incorporating data from external sources.

For example, here are some features that can be extracted from the 'Date' column:
1) Day of Week
2) Day or month
3) Month
4) Year
5) Weekend/Weekday
6) Month/Quater End

In [ ]:
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

In [ ]:
merged_df['Day'] = merged_df.Date.dt.day
merged_df['Month'] = merged_df.Date.dt.month
merged_df['Year'] = merged_df.Date.dt.year

In [ ]:
merged_test_df['Day'] = merged_test_df.Date.dt.day
merged_test_df['Month'] = merged_test_df.Date.dt.month
merged_test_df['Year'] = merged_test_df.Date.dt.year

In [ ]:
merged_df

In [ ]:
sns.barplot(data=merged_df.sample(40000), x='Year', y='Sales')

In [ ]:
sns.barplot(data=merged_df.sample(40000), x='Month', y='Sales', hue='Month')

In [ ]:
plt.figure(figsize=[10, 5])
sns.barplot(data=merged_df.sample(40000), x='Day', y='Sales', hue='Day')

We can also add some other diffenrent columns but these can be important like:
1) if it rained or not on that day
2) what was the temperature
3) population near the store

these can make a significant impact on our model prediciton.

# Step-6> Creating training/test/validation split and prepare the data for training

# Step-6-A> Train/Test/Validation Split

The data already contains a test set, which contains over one month of data after the end of the training set. We can apply a similar strategy to create a validation set. We'll the last 25% of rows for the validation set, after ordering by date


In [ ]:
len(merged_df)

In [ ]:
train_size = int(0.75 * len(merged_df))
train_size

In [ ]:
sorted_df = merged_df.sort_values('Date')
train_df, val_df = sorted_df[:train_size], sorted_df[train_size:]

In [ ]:
len(train_df), len(val_df)

In [ ]:
train_df.Date.min(), train_df.Date.max()

In [ ]:
val_df.Date.min(), val_df.Date.max()

# Step-6-B> Separating Input and Target Columns

Let's also identify input and target columns. *Note that we can't use the no. of customers as an input, because this*
information isn't available beforehand. Also, we needn't use all the available columns, we can start out with just a small
subset.

In [ ]:
train_df.columns

In [ ]:
input_cols = ['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'StoreType', 'Assortment', 'Day', 'Month', 'Year']

target_cols = 'Sales'

**Curse of Dimensionality** in machine learning refers to the problem that arises when the number of features increases, causing data to become sparse and distances between data points to lose meaning, which makes learning patterns difficult and reduces model performance.

In my dataset, the *store* feature is a categorical column with **1115 unique values**. When such a feature is encoded (for example, using one-hot encoding), it can create a very high-dimensional feature space, increasing sparsity and computational cost, and potentially leading to overfitting—this is a practical example of the curse of dimensionality.

# Step-6-C> Separating Input & Output Columns

In [ ]:
merged_df[input_cols].nunique()

In [ ]:
train_inputs = train_df[input_cols].copy()
train_targets = train_df[target_cols].copy()

val_inputs = val_df[input_cols].copy()
val_targets = val_df[target_cols].copy()

test_inputs = merged_test_df[input_cols].copy()
#test data does not have targets

Note: Some columns can be treated as both numeric and categorical, and it's up to you how you deal with them

In [ ]:
numeric_cols = ['Store', 'Day', 'Month', 'Year']
cat_cols = ['DayOfWeek', 'Promo', 'StateHoliday', 'StoreType', 'Assortment']

In [ ]:
train_inputs

# Step-7-D> Imputation, Scaling and Encode

Imputing missing data from numeric columns

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy='mean').fit(train_inputs[numeric_cols])

In [ ]:
train_inputs[numeric_cols] = imputer.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = imputer.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer.transform(test_inputs[numeric_cols])

Note that this step wasn't necessary for the store sales dataset, as there were no null values. Also, we can apply a differentimputation strategy to different columns depending on their distributions (e.g. mean for normally distribute and median forexponentially distributed).

# Step-7-E> Scaling the inputs

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
scaler = MinMaxScaler().fit(train_inputs[numeric_cols])

In [ ]:
train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
val_inputs[numeric_cols] = scaler.transform(val_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

In [ ]:
train_inputs

# Step-7-F> Encoding the Categorical Columns

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
train_df['DayOfWeek']

In [ ]:
train_df[cat_cols].info()

In [ ]:
for col in cat_cols:
    print(col, train_inputs[col].map(type).value_counts())

In [ ]:
train_inputs['StateHoliday'] = train_inputs['StateHoliday'].astype(str)
test_inputs['StateHoliday'] = test_inputs['StateHoliday'].astype(str)
val_inputs['StateHoliday'] = val_inputs['StateHoliday'].astype(str)

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(train_inputs[cat_cols])
encoded_cols = list(encoder.get_feature_names_out(cat_cols))


In [ ]:
encoded_cols

now creating the perfect data which in be direclty fit into the model

In [ ]:
train_inputs[encoded_cols] = encoder.transform(train_inputs[cat_cols])
val_inputs[encoded_cols] = encoder.transform(val_inputs[cat_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[cat_cols])


In [ ]:
for col in train_inputs.columns:
    print(col, train_inputs.columns.map(type).value_counts())

In [ ]:
X_train = train_inputs[numeric_cols+encoded_cols]
X_test = test_inputs[numeric_cols+encoded_cols]
X_val= val_inputs[numeric_cols+encoded_cols]

In [ ]:
for col in X_train.columns:
    print(col, X_train[col].map(type).value_counts())

In [ ]:
from sklearn.metrics import mean_absolute_error

***Tree Based Model***

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model=DecisionTreeRegressor(random_state=54,
                            max_depth=35,
                            min_impurity_decrease=1e-6).fit(X_train, train_targets)

In [ ]:
train_preds = model.predict(X_train)
val_preds = model.predict(X_val)

In [ ]:
model.score(X_train, train_targets), model.score(X_val, val_targets), mean_absolute_error(train_targets, train_preds), mean_absolute_error(val_targets, val_preds)

In [ ]:
plt.figure(figsize=(30, 10))
plot_tree(model, max_depth=2, feature_names=X_test.columns, fontsize=11, filled=True)

In [ ]:
#{'train_rmse': 474.38862159017947,
# 'val_rmse': 1373.1518287790345,
# 'train_MAE': 27 6.7373814057926,
# 'val_MAE': 891.8270944300751}
#got these with no hyperparametring max_depth = 61, n_estimators=100
rf = RandomForestRegressor(random_state=54, 
                                n_jobs=-1, 
                                verbose=1,
                                max_depth=15,
                                n_estimators=50)

In [ ]:
rf.fit(X_train, train_targets)

In [ ]:
train_preds = rf.predict(X_train)
train_preds

In [ ]:
val_preds = rf.predict(X_val)

In [ ]:
rf.score(X_train, train_targets), rf.score(X_val, val_targets), mean_absolute_error(train_targets, train_preds), mean_absolute_error(val_targets, val_preds)

## Neural Network Model

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(383)
import torch.optim as optim

#### Create CustomDataset Class for **Dataset**

In [ ]:
# MinMaxScaler expects a 2D input. We reshape the targets to shape (-1, 1) for fit & transform.
train_targets_2d = train_targets.values.reshape(-1, 1)
val_targets_2d = val_targets.values.reshape(-1, 1)

mmxencoder = MinMaxScaler().fit(train_targets_2d)

# Overwrite train_targets and val_targets with the scaled values
train_targets_scaled = mmxencoder.transform(train_targets_2d)
val_targets_scaled = mmxencoder.transform(val_targets_2d)


In [ ]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features.values).to(torch.float32)
        
        # Handle both pandas Series and numpy arrays (whether 1D or 2D)
        if hasattr(labels, 'values'):
            self.label = torch.tensor(labels.values).to(torch.float32)
        else:
            self.label = torch.tensor(labels).to(torch.float32)
            
        # Ensure label tensor is 2-dimensional (shape N x 1)
        if self.label.ndim == 1:
            self.label = self.label.unsqueeze(1)
        
    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.label[index]


In [ ]:
train_dataset = CustomDataset(X_train, train_targets_scaled)
val_dataset = CustomDataset(X_val, val_targets_scaled)

#### Create train and val DataLoader objects

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=436, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=436, shuffle=False)

### Create NN model

In [ ]:
len(X_train.columns)

In [ ]:
class RossmannNN(nn.Module):
    def __init__(self, num_of_features):
        super().__init__()
        self.model = nn.Sequential(
            
            nn.Linear(num_of_features, 12),
            nn.BatchNorm1d(12),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            
            nn.Linear(12, 7),
            nn.BatchNorm1d(7),
            nn.ReLU(),
            #nn.Dropout(p=0.2),
            
            nn.Linear(7,1)            
        )
        
    def forward(self, data):
        return self.model(data)

In [ ]:
epochs = 20
learning_rate = 0.001

In [ ]:
nnModel = RossmannNN(len(X_train.columns))
loss_criterion = nn.MSELoss()
optimizer = optim.Adam(nnModel.parameters(), lr=learning_rate, weight_decay=1e-4)

In [ ]:
# total number of batches
len(train_loader)

In [ ]:
nnModel.train() # set the model to training mode

for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

    
        y_pred = nnModel(batch_features)
        loss = loss_criterion(y_pred, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_epoch_loss = total_epoch_loss + loss.item()
        
    average_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch+1} | Loss: {average_loss}')

### Evaluate the Model

In [ ]:
nnModel.eval()

total = 0
loss_cal = 0

with torch.no_grad():
    for batch_features, batch_labels in val_loader:
        output = nnModel(batch_features)
        
        output_np = output.numpy()
        labels_np = batch_labels.numpy()
        
        output_org = mmxencoder.inverse_transform(output_np)
        labels_org = mmxencoder.inverse_transform(labels_np)
        
        total += batch_labels.shape[0]
        
        loss_cal = loss_cal + abs(output_org - labels_org).sum().item()
        
print(f'Accuracy on test data: {loss_cal/total}')


In [ ]:
nnModel.eval()

total = 0
loss_cal = 0

with torch.no_grad():
    for batch_features, batch_labels in train_loader:
        output = nnModel(batch_features)
        
        output_np = output.numpy()
        labels_np = batch_labels.numpy()
        
        output_org = mmxencoder.inverse_transform(output_np)
        labels_org = mmxencoder.inverse_transform(labels_np)
        
        total += batch_labels.shape[0]
        
        loss_cal = loss_cal + abs(output_org - labels_org).sum().item()
        
print(f'Accuracy on train data: {loss_cal/total}')
# Accuracy on train data: 0.05285828513670347
# Accuracy on train data: 0.059868708924065414 -> without overfitting code

# Step-11: Interpret Models, Study Individual Predictions & Present Your Findings

**Feature Importance**

You'll need to explain why your model returns a particular result. Most scikit-learn models offer some kind of "feature importance" score.

In [ ]:
X_train.columns

In [ ]:
rf.feature_importances_

In [ ]:
importance_df = pd.DataFrame({
    'columns': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values(by='importance', ascending=False)

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(data=importance_df, x='importance', y='columns', hue="columns")

In [ ]:
x = np.arange(len(train_targets))   # SAME x-axis

plt.figure(figsize=(20, 5))
plt.plot(x[:1000], train_targets[:1000], label='Actual', alpha=0.6)
plt.plot(x[:1000], train_preds[:1000], label='Predicted', alpha =0.6)

plt.xlabel('Sample Index')
plt.ylabel('sales Value')
plt.title('Actual vs Predicted')
plt.legend()
plt.show()                                                                                  

In [ ]:
actual_preds_df = pd.DataFrame({
    'Index': range(len(train_targets)),
    'Actual': train_targets,
    'Predicted': train_preds
})

In [ ]:
fig = px.line(
    actual_preds_df[:5000],
    x='Index',
    y=['Actual', 'Predicted'],
    labels={'value': 'Value', 'variable': 'Type'}
)
fig.update_layout(width=1200, height=500, hovermode='x unified')
fig.update_traces(opacity=0.6)
fig.show()

In [ ]:
fig = px.line(x=range(len(train_targets[:2000])), y=[list(train_targets[:2000]), list(train_preds[:2000])], labels={'value':'Sales value', 'variable':'type'})
fig.update_traces(opacity=0.4)
fig.update_layout(hovermode='x unified')